# Unity Catalog Functions for E-commerce

This notebook creates reusable Unity Catalog functions (UDFs) that can be called by AI agents and applications to:
* Retrieve company policy details
* Query customer service history

These functions enable consistent data access patterns across the organization and can be integrated into agentic AI workflows.

## Function 1: Get Return Policy

Creates a table-valued function that retrieves policy details from the policies table. This function can be used by AI agents to answer customer questions about company policies.

In [0]:
# Create a table-valued function in Unity Catalog
# This function retrieves specific policy details from the policies table
spark.sql(f"""
-- Create a function to retrieve company policy details
CREATE OR REPLACE FUNCTION agentic_ai.ecommerce.get_return_policy(
    policy_name STRING 
    COMMENT 'Policy name to return. Example policies: Account Cancellation Policy, Exchange Policy, Refund Policy, Warranty Policy, Privacy Policy, Return Policy'
    )
    RETURNS TABLE (
    policy           STRING,
    policy_details   STRING,
    last_updated     DATE
    )
    COMMENT 'Returns the details of the Return Policy'
    LANGUAGE SQL
    RETURN (
    SELECT
    policy,
    policy_details,
    last_updated
    FROM agentic_ai.ecommerce.policies
    WHERE policy = policy_name
    LIMIT 1
    );
    """)

### Test: Retrieve Exchange Policy

Query the function to validate it returns the correct policy details.

In [0]:
%sql
select * from agentic_ai.ecommerce.get_return_policy("Exchange Policy")

## Function 2: Get Service History

Creates a table-valued function that retrieves customer service history by email address. Returns the count of returns in the last 12 months grouped by issue category.

In [0]:
# Create a table-valued function to query customer service history
# Aggregates return counts by issue category for a given customer email
spark.sql(f"""
-- Create a function to get service history by user
CREATE OR REPLACE FUNCTION agentic_ai.ecommerce.get_service_history(
    user_email STRING 
    COMMENT 'User email to retrieve order history'
    )
    RETURNS TABLE (
    returns_last_12_months INT,
    issue_category STRING, 
    todays_date DATE
    )
    COMMENT 'This takes the user_name of a customer as an input and returns the number of returns and the issue category'
    LANGUAGE SQL
    RETURN(
    SELECT count(*) as returns_last_12_months, issue_category, now() as todays_date
    FROM agentic_ai.ecommerce.customer_service_data 
    WHERE email = user_email
    GROUP BY issue_category
    );""")

### Test: Retrieve Customer Service History

Query the function with a sample customer email to validate it returns service history data.

In [0]:
%sql
select * from agentic_ai.ecommerce.get_service_history("nicolas.pelaez@example.com")

## Summary

This notebook created two Unity Catalog functions in the `agentic_ai.ecommerce` schema:

1. **`get_return_policy(policy_name)`** - Returns policy details for a given policy name
   * Input: Policy name (string)
   * Output: Policy, policy_details, last_updated date

2. **`get_service_history(user_email)`** - Returns customer service history for a given email
   * Input: Customer email (string)
   * Output: Returns count in last 12 months, issue category, today's date

These functions can now be:
* Called from SQL queries and notebooks
* Used by AI agents to answer customer questions
* Integrated into agentic workflows for customer service automation